
# 🏙️ Toronto Airbnb Analysis
## Week 1 — Load & Explore

**Business question:** What drives Airbnb pricing in Toronto, and which neighbourhoods offer the best opportunity for hosts?

**This notebook:** First look at the raw data. No cleaning yet — just understanding what we have and what's broken.

**Data source:** [Inside Airbnb](http://insideairbnb.com/get-the-data/) → Toronto → `listings.csv.gz`

---

## 0. Setup
Install dependencies and import libraries. Run this cell first.

In [51]:
# Run this once if you haven't installed pandas yet
# !pip install pandas

import pandas as pd

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load the Data

We load the raw CSV directly from the `.gz` file — no need to unzip it manually. Pandas handles it automatically.

> **Make sure** `listings.csv.gz` is in the same folder as this notebook.

In [52]:
df = pd.read_csv('listings.csv', low_memory=False)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 21,391
Columns : 79


## 2. What Columns Do We Have?

Before doing anything, we need to know what fields exist. Some will be useful, most won't.

In [53]:
print(f"All {df.shape[1]} columns:\n")
for col in df.columns:
    print(f"  {col}")

All 79 columns:

  id
  listing_url
  scrape_id
  last_scraped
  source
  name
  description
  neighborhood_overview
  picture_url
  host_id
  host_url
  host_name
  host_since
  host_location
  host_about
  host_response_time
  host_response_rate
  host_acceptance_rate
  host_is_superhost
  host_thumbnail_url
  host_picture_url
  host_neighbourhood
  host_listings_count
  host_total_listings_count
  host_verifications
  host_has_profile_pic
  host_identity_verified
  neighbourhood
  neighbourhood_cleansed
  neighbourhood_group_cleansed
  latitude
  longitude
  property_type
  room_type
  accommodates
  bathrooms
  bathrooms_text
  bedrooms
  beds
  amenities
  price
  minimum_nights
  maximum_nights
  minimum_minimum_nights
  maximum_minimum_nights
  minimum_maximum_nights
  maximum_maximum_nights
  minimum_nights_avg_ntm
  maximum_nights_avg_ntm
  calendar_updated
  has_availability
  availability_30
  availability_60
  availability_90
  availability_365
  calendar_last_scraped
  num

## 3. Data Types

This tells us what Pandas *thinks* each column is. Watch for columns that should be numbers but show up as `object` (string). That's a cleaning problem we'll fix in Week 2.

> **Hint:** The `price` column will almost certainly show as `object` — it comes in as `$150.00`, not `150.0`.

In [54]:
print(df.dtypes.to_string())

id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
name                                             object
description                                      object
neighborhood_overview                            object
picture_url                                      object
host_id                                           int64
host_url                                         object
host_name                                        object
host_since                                       object
host_location                                    object
host_about                                       object
host_response_time                               object
host_response_rate                               object
host_acceptance_rate                            

## 4. Missing Values

Every real dataset has gaps. Here we find which columns have missing data and how bad it is. Columns missing more than 50% of values are usually dropped. Columns missing 5–20% need a decision — drop the rows, or fill with a sensible default.

In [55]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values found.")
else:
    print(f"{'Column':<45} {'Missing':>8}  {'% Missing':>10}")
    print("-" * 67)
    for col, count in missing.items():
        pct = (count / len(df)) * 100
        flag = " ⚠️ HIGH" if pct > 40 else ""
        print(f"  {col:<43} {count:>8,}  {pct:>9.1f}%{flag}")

Column                                         Missing   % Missing
-------------------------------------------------------------------
  calendar_updated                              21,391      100.0% ⚠️ HIGH
  neighbourhood_group_cleansed                  21,391      100.0% ⚠️ HIGH
  host_neighbourhood                            13,715       64.1% ⚠️ HIGH
  neighborhood_overview                         11,570       54.1% ⚠️ HIGH
  neighbourhood                                 11,570       54.1% ⚠️ HIGH
  host_about                                    11,542       54.0% ⚠️ HIGH
  license                                       11,013       51.5% ⚠️ HIGH
  host_response_time                             6,774       31.7%
  host_response_rate                             6,774       31.7%
  beds                                           6,003       28.1%
  bathrooms                                      6,000       28.0%
  estimated_revenue_l365d                        5,993       28.0%
  pri

## 5. The Price Column Problem

This is intentionally broken — and fixing it will be your first real cleaning task in Week 2.

The `price` column comes in as a string like `"$150.00"`. You can't do math on a string. We need to strip the `$` and `,` characters and convert it to a float.

In [56]:
if 'price' in df.columns:
    print(f"Data type  : {df['price'].dtype}")
    print(f"\nSample values:")
    for val in df['price'].dropna().head(8).values:
        print(f"  {val}")
    print("\n⚠️  This is a string with '$' — you cannot do math on it yet.")
    print("   You'll fix this in Week 2.")

Data type  : object

Sample values:
  $112.00
  $80.00
  $94.00
  $120.00
  $144.00
  $50.00
  $48.00
  $60.00

⚠️  This is a string with '$' — you cannot do math on it yet.
   You'll fix this in Week 2.


## 6. Preview the Key Columns

Out of all the columns, these are the ones we'll use most throughout the project. Let's see the first few rows of just these.

In [57]:
key_cols = [
    'id', 'name', 'neighbourhood_cleansed', 'room_type',
    'price', 'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'availability_365', 'host_id'
]

# Only keep columns that actually exist in this dataset
existing = [c for c in key_cols if c in df.columns]

df[existing].head(10)

,id,name,neighbourhood_cleansed,room_type,price,minimum_nights,number_of_reviews,review_scores_rating,availability_365,host_id
0,1419,Beautiful home in amazing area!,Little Portugal,Entire home/apt,NaN,28,6,5.00,0,1565
1,8077,Downtown Harbourfront Private Room,Waterfront Communities-The Island,Private room,NaN,180,169,4.84,0,22795
2,26654,"World Class @ CN Tower, convention centre, The...",Waterfront Communities-The Island,Entire home/apt,$112.00,28,45,4.80,0,113345
3,27423,Executive Studio Unit- Ideal for One Person,South Riverdale,Entire home/apt,NaN,365,32,4.94,0,118124
4,30931,Downtown Toronto - Waterview Condo,Waterfront Communities-The Island,Entire home/apt,NaN,180,1,5.00,0,22795
5,40456,Downtown- King Size Bed and Parking,South Parkdale,Entire home/apt,NaN,750,113,4.64,364,174063
6,40701,Bright Beaches loft close to Queen and the lake,The Beaches,Entire home/apt,$80.00,28,11,4.64,45,175687
7,44452,Yonge & Bloor Studio Skyline,Rosedale-Moore Park,Entire home/apt,$94.00,28,67,4.18,243,195095
8,45399,Fountain View Studio - Eaton center,Bay Street Corridor,Entire home/apt,$120.00,28,89,4.17,307,195095
9,45893,Yonge & Bloor Lakeview Master BR,Rosedale-Moore Park,Private room,NaN,28,24,4.40,245,195095


## 7. Neighbourhood Breakdown

How many listings does each neighbourhood have? This tells us which areas are most active in the Toronto Airbnb market.

In [58]:
if 'neighbourhood_cleansed' in df.columns:
    neighbourhoods = df['neighbourhood_cleansed'].value_counts()
    print(f"Total unique neighbourhoods: {len(neighbourhoods)}\n")
    print(f"{'Neighbourhood':<40} {'Listings':>10}")
    print("-" * 52)
    for name, count in neighbourhoods.head(15).items():
        print(f"  {name:<38} {count:>10,}")

Total unique neighbourhoods: 140

Neighbourhood                              Listings
----------------------------------------------------
  Waterfront Communities-The Island           3,523
  Niagara                                       872
  Annex                                         685
  Church-Yonge Corridor                         638
  Kensington-Chinatown                          558
  Dovercourt-Wallace Emerson-Junction           525
  Trinity-Bellwoods                             524
  Bay Street Corridor                           520
  Moss Park                                     513
  Willowdale East                               441
  Little Portugal                               403
  South Riverdale                               383
  Palmerston-Little Italy                       342
  South Parkdale                                282
  York University Heights                       272


## 8. Room Type Breakdown

Airbnb has four room types. The split between them affects pricing significantly — entire homes command higher prices than shared rooms.

In [59]:
if 'room_type' in df.columns:
    room_counts = df['room_type'].value_counts()
    print(f"{'Room Type':<35} {'Count':>8}  {'% of Total':>10}")
    print("-" * 57)
    for rtype, count in room_counts.items():
        pct = (count / len(df)) * 100
        bar = "█" * int(pct / 2)
        print(f"  {rtype:<33} {count:>8,}  {pct:>9.1f}%  {bar}")

Room Type                              Count  % of Total
---------------------------------------------------------
  Entire home/apt                     14,163       66.2%  █████████████████████████████████
  Private room                         7,165       33.5%  ████████████████
  Shared room                             55        0.3%  
  Hotel room                               8        0.0%  


## 9. Basic Statistics on Numeric Columns

`describe()` gives us count, mean, min, max, and quartiles for every numeric column. This is a quick way to spot outliers — for example, a `minimum_nights` value of 1000 is obviously wrong.

In [60]:
numeric_cols = [
    'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'availability_365'
]

existing_numeric = [c for c in numeric_cols if c in df.columns]
df[existing_numeric].describe().round(2)

,minimum_nights,number_of_reviews,review_scores_rating,availability_365
count,21391.00,21391.00,16254.00,21391.00
mean,25.02,28.00,4.79,155.59
std,39.12,57.36,0.37,131.09
min,1.00,0.00,1.00,0.00
25%,3.00,1.00,4.74,22.00
50%,28.00,6.00,4.89,135.00
75%,28.00,31.00,5.00,270.00
max,1125.00,1208.00,5.00,365.00


## 10. ✅ Week 1 Homework

Before moving to Week 2, write down your answers to these questions. Open a text file or a new markdown cell below and record what you found. These notes become your project README later.

1. How many listings are in the dataset?
2. Which neighbourhood has the most listings?
3. What percentage of listings are `Entire home/apt`?
4. Name 3 columns with significant missing data — what percentage is missing?
5. Why can't you calculate the average price right now?
6. What's the maximum value of `minimum_nights`? Does it seem realistic?

---

> **Week 2 preview:** You'll fix the price column, drop bad rows, handle missing values, and export a clean CSV. That clean CSV becomes the foundation for every chart and query you build after.

In [61]:
# Your notes go here — add your answers as print statements or just read them from the outputs above
print("My Week 1 findings:")
print(f"  Total listings      : {len(df):,}")
print(f"  Total columns       : {df.shape[1]}")
print(f"  Neighbourhoods      : {df['neighbourhood_cleansed'].nunique() if 'neighbourhood_cleansed' in df.columns else 'N/A'}")
print(f"  Room types          : {df['room_type'].nunique() if 'room_type' in df.columns else 'N/A'}")
print(f"  Price column type   : {df['price'].dtype if 'price' in df.columns else 'N/A'}  ← needs fixing in Week 2")

My Week 1 findings:
  Total listings      : 21,391
  Total columns       : 79
  Neighbourhoods      : 140
  Room types          : 4
  Price column type   : object  ← needs fixing in Week 2


Answers:
1. Total listings = 21,391
2. Waterfront Communities-The Island has the most listings, with 3,523 listings.
3. 66.2% of listings are entire home/apt.
4. neighbourhood_group_cleansed, and calendar_updated are 2 columns with significant missing data. 100% of them are missing. The next column with highest missing data is host_neighbourhood, with 64.1% missing
5. I cannot calculate average price right now because the column is currently an 'object' dtype.
6. The maximum value of minimum_nights is 1125 nights, roughly 3 years. It is not realistic, as the medium and mean points towards roughly a month.
